4) Chain of Density(CoD) 요약

In [ ]:
# [목적] Chain of Density 요약에 필요한 환경, LangChain 객체, PDF 문서를 준비합니다.
# API 환경 변수를 불러오고 LangSmith 연결 객체를 만든 뒤 PDF를 페이지별 문서로 읽습니다.
# 준비된 docs는 이후 정보 밀도를 반복해서 높이는 CoD 체인의 원문으로 사용합니다.
from dotenv import load_dotenv

import textwrap
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import SimpleJsonOutputParser
from langchain_community.document_loaders import PyPDFLoader

# .env 파일에 저장된 OpenAI·LangSmith API 키 등의 환경 변수를 불러옵니다.
load_dotenv()

# Client는 LangSmith Hub에 저장된 프롬프트를 조회하는 연결 객체입니다.
client = Client()

# PyPDFLoader는 PDF의 각 페이지를 본문과 메타데이터가 담긴 Document로 변환합니다.
loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

In [ ]:
# [목적] LangSmith Hub에서 Chain of Density 전용 프롬프트를 불러와 내용을 확인합니다.
# 공개 프롬프트를 가져온 뒤 입력 변수와 요약 지시문을 읽기 쉬운 형태로 출력합니다.
# 확인한 cod_prompt는 원문의 정보 밀도를 단계적으로 높이는 체인에 사용합니다.
cod_prompt = client.pull_prompt(
    "teddynote/chain-of-density-prompt",
    dangerously_pull_public_prompt=True,
)

cod_prompt.pretty_print()

In [ ]:
# [목적] CoD 프롬프트에 전달할 입력값과 생략 가능한 옵션의 기본값을 정의합니다.
# 실행 시 받은 딕셔너리에서 값을 꺼내고, 값이 없으면 문서 유형·길이·반복 횟수 등을 기본값으로 채웁니다.
# 정리된 입력은 프롬프트가 항상 필요한 형식을 받도록 해 체인을 안정적으로 실행하게 합니다.
# 각 lambda는 체인 입력 d에서 값을 읽는 짧은 함수이며 content만 필수로 받습니다.
cod_chain_inputs = {
    "content": lambda d: d.get("content"),
    "content_category": lambda d: d.get("content_category", "Article"),
    "entity_range": lambda d: d.get("entity_range", "1-3"),
    "max_words": lambda d: int(d.get("max_words", 80)),
    "iterations": lambda d: int(d.get("iterations", 5)),
}

# 위 입력 구조와 연결할 Chain of Density 공개 프롬프트를 가져옵니다.
cod_prompt = client.pull_prompt(
    "teddynote/chain-of-density-prompt",
    dangerously_pull_public_prompt=True
)

In [ ]:
# [목적] 입력 정리부터 반복 요약과 JSON 변환까지 수행하는 Chain of Density 체인을 만듭니다.
# 입력값을 프롬프트와 모델에 전달하고, 모델의 JSON 응답을 Python 목록과 딕셔너리로 변환합니다.
# 별도 체인은 반복 결과 중 마지막 denser_summary만 꺼내 최종 요약이 필요할 때 사용합니다.
cod_chain = (  # Chain of Density 체인 생성
    cod_chain_inputs
    | cod_prompt
    | ChatOpenAI(temperature=0, model="gpt-4o-mini")
    # SimpleJsonOutputParser는 모델이 만든 JSON 문자열을 Python 객체로 변환합니다.
    | SimpleJsonOutputParser()
)

# 전체 반복이 끝난 결과의 마지막 항목에서 가장 밀도 높은 요약만 추출합니다.
cod_final_summary_chain = cod_chain | (
    # output[-1]은 반복해서 생성된 요약 목록의 마지막 결과를 뜻합니다.
    lambda output: output[-1].get(
        "denser_summary",
        '오류: 마지막 딕셔너리에 "denser_summary" 키가 없습니다',
    )
)

In [ ]:
# [목적] PDF의 두 번째 페이지 본문을 CoD 요약에 사용할 원문으로 선택합니다.
# Document의 page_content에서 실제 텍스트를 꺼내 content에 저장하고 내용을 확인합니다.
# 이 문자열은 다음 셀에서 cod_chain의 필수 입력값으로 전달됩니다.
content = docs[1].page_content
print(content)

In [ ]:
# [목적] CoD 체인을 스트리밍으로 실행하고 밀도가 높아지는 각 단계의 요약을 정리해 출력합니다.
# 생성 중인 JSON 결과를 계속 갱신한 뒤, 각 요약에서 새 엔티티와 본문을 꺼내 순서대로 표시합니다.
# 마지막 반복의 summary는 모든 핵심 정보가 반영된 최종 요약으로 사용합니다.
# 결과는 반복별 딕셔너리가 담기는 목록이며, 처음에는 빈 목록으로 준비합니다.
results: list[dict[str, str]] = []

# stream은 모델 응답이 완성되는 동안 파싱 가능한 최신 JSON 결과를 차례로 전달합니다.
for partial_json in cod_chain.stream(
    {"content": content, "content_category": "Article"}
):
    # 새 조각이 도착할 때마다 현재까지 파싱된 전체 결과로 갱신합니다.
    results = partial_json

    # 현재 결과를 같은 줄에 출력(캐리지 리턴을 사용하여 이전 출력을 덮어씀)
    print(results, end="\r", flush=True)

# 총 요약본 수 계산
total_summaries = len(results)
print("\n")

# 밀도를 높인 순서대로 각 요약본과 새롭게 추가된 핵심 엔티티를 확인합니다.
i = 1
for cod in results:
    # 세미콜론으로 구분된 missing_entities를 읽기 쉬운 쉼표 목록으로 바꿉니다.
    added_entities = ", ".join(
        [
            ent.strip()
            for ent in cod.get(
                "missing_entities", 'ERR: "missing_entities" key not found'
            ).split(";")
        ]
    )

    # denser_summary에는 현재 반복에서 더 많은 핵심 정보를 반영한 요약이 들어 있습니다.
    summary = cod.get("denser_summary", 'ERR: missing key "denser_summary"')

    print(
        # 요약 정보 출력(번호, 총 개수, 추가된 엔티티)
        f"### CoD Summary {i}/{total_summaries}, 추가된 엔티티(entity): {added_entities}"
        + "\n"
    )

    # 요약 내용을 80자 너비로 줄바꿈하여 출력
    print(textwrap.fill(summary, width=80) + "\n")
    i += 1

print("\n============= [최종 요약] =============\n")
print(summary)

In [ ]:
# [목적] CoD 반복 과정에서 완성된 최종 고밀도 요약을 다시 출력합니다.
# 앞 셀의 마지막 반복에서 저장된 summary를 사용해 최종 결과만 간단히 확인합니다.P
# 이 값은 화면 확인뿐 아니라 이후 저장이나 후속 분석에도 활용할 수 있습니다.
print(summary)